# Task 1


In [8]:
!pip -q install -U transformers peft accelerate datasets

In [11]:
import torch
import transformers
import peft
import datasets

print("Installation successful!")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

Installation successful!
GPU: Tesla T4


In [10]:
import json, random, time, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, PeftModel
SEED=42
random.seed(SEED); torch.manual_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU: Tesla T4


In [3]:
intents=[("reminder","remind me to {item}"),("task","please {item}"),("shopping","add {item} to my shopping list"),("note","make a note that I need to {item}"),("schedule","schedule time to {item}")]
items=["submit my assignment","call the dentist","buy milk","finish the project report","email my professor","renew my library card","book a train ticket","pay the electricity bill","prepare the presentation","water the plants","pick up the parcel","review the lecture notes","clean the desk","update my resume","back up my files","order printer paper","send the meeting minutes","check the application status","charge my headphones","return the borrowed book","complete the Python exercise","buy vegetables","confirm the appointment","organise the project folder","read chapter three"]
priorities=["low","medium","high"]
wrappers=["{base}. Priority is {priority}.","Could you help me? {base}, mark it {priority} priority.","{base} — this is {priority} priority.","Please record this: {base}. Set priority to {priority}.","I need to remember this: {base}. It has {priority} priority."]
data=[]
for intent,tmpl in intents:
 for item in items:
  for priority in priorities:
   for wrap in wrappers:
    data.append({"prompt":wrap.format(base=tmpl.format(item=item),priority=priority),
      "gold":{"intent":intent,"item":item,"quantity":1,"priority":priority}})
random.shuffle(data)
test=data[:60]; train=data[60:660]
print("Train",len(train),"held-out test",len(test))

Train 600 held-out test 60


In [4]:
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
tok=AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None: tok.pad_token=tok.eos_token
system="Convert the request into exactly one JSON object with keys intent, item, quantity, priority. Use intent from reminder, task, shopping, note, schedule. quantity must be an integer. priority must be low, medium, or high. Output JSON only."
def chat(prompt, answer=None):
 m=[{"role":"system","content":system},{"role":"user","content":prompt}]
 if answer is not None: m.append({"role":"assistant","content":json.dumps(answer)})
 return tok.apply_chat_template(m,tokenize=False,add_generation_prompt=False)
def encode(x): return tok(chat(x["prompt"],x["gold"]),truncation=True,max_length=256)
ds=Dataset.from_list(train).map(encode,remove_columns=["prompt","gold"])

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

In [5]:
base=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,device_map="auto")
base.config.use_cache=False
cfg=LoraConfig(r=8,lora_alpha=16,lora_dropout=0.05,target_modules=["q_proj","v_proj"],bias="none",task_type="CAUSAL_LM")
model=get_peft_model(base,cfg); model.print_trainable_parameters()
args=TrainingArguments(output_dir="qwen_lora_output",per_device_train_batch_size=4,gradient_accumulation_steps=4,num_train_epochs=2,learning_rate=2e-4,fp16=torch.cuda.is_available(),logging_steps=20,save_strategy="epoch",report_to="none",seed=SEED)
trainer=Trainer(model=model,args=args,train_dataset=ds,data_collator=DataCollatorForLanguageModeling(tok,mlm=False))
t=time.time(); trainer.train(); print("Training minutes:",round((time.time()-t)/60,2))
trainer.model.save_pretrained("qwen_lora_adapter"); tok.save_pretrained("qwen_lora_adapter")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/import_utils.py:195: UserWarning: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported. torchao support in PEFT is disabled; if you want to use a torchao-quantized model with PEFT, please upgrade torchao.
  warnings.warn(


trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


Step,Training Loss
20,2.453930
40,1.096376
60,0.538008


Training minutes: 0.94


('qwen_lora_adapter/tokenizer_config.json',
 'qwen_lora_adapter/chat_template.jinja',
 'qwen_lora_adapter/tokenizer.json')

In [13]:

def generate(m, prompt):
    inputs = tok.apply_chat_template(
        [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    device = next(m.parameters()).device
    input_ids = inputs["input_ids"].to(device)

    with torch.no_grad():
        output = m.generate(
            input_ids,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tok.eos_token_id
        )

    generated_tokens = output[0][input_ids.shape[-1]:]

    return tok.decode(
        generated_tokens,
        skip_special_tokens=True
    )


In [15]:
base.config.use_cache = True

base_metrics, base_rows = evaluate(base, "base")

print("BASE MODEL RESULTS:")
print(base_metrics)

BASE MODEL RESULTS:
{'model': 'base', 'n': 60, 'json_valid_rate': 1.0, 'exact_match_accuracy': 0.7666666666666667}


In [16]:
ft = PeftModel.from_pretrained(
    base,
    "qwen_lora_adapter"
)

ft_metrics, ft_rows = evaluate(ft, "lora")

print("FINE-TUNED MODEL RESULTS:")
print(ft_metrics)

/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:342: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


FINE-TUNED MODEL RESULTS:
{'model': 'lora', 'n': 60, 'json_valid_rate': 1.0, 'exact_match_accuracy': 0.7833333333333333}


In [17]:
print("Base:", base_metrics)
print("Fine-tuned:", ft_metrics)

Base: {'model': 'base', 'n': 60, 'json_valid_rate': 1.0, 'exact_match_accuracy': 0.7666666666666667}
Fine-tuned: {'model': 'lora', 'n': 60, 'json_valid_rate': 1.0, 'exact_match_accuracy': 0.7833333333333333}


In [18]:
results = {
    "base": base_metrics,
    "finetuned": ft_metrics
}

import json

with open("task1_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Results saved successfully!")

Results saved successfully!


In [19]:
failures = [r for r in ft_rows if not r["exact"]]

print("Total failed examples:", len(failures))

for i, r in enumerate(failures[:3], 1):
    print("\n========== FAILURE", i, "==========")
    print("INPUT:", r["prompt"])
    print("EXPECTED:", r["gold"])
    print("MODEL OUTPUT:", r["raw"])
    print("PARSED OUTPUT:", r["parsed"])

Total failed examples: 13

========== FAILURE 1 ==========
INPUT: make a note that I need to order printer paper. Priority is low.
EXPECTED: {'intent': 'note', 'item': 'order printer paper', 'quantity': 1, 'priority': 'low'}
MODEL OUTPUT: {"intent": "note", "item": "printer paper", "quantity": 1, "priority": "low"}
PARSED OUTPUT: {'intent': 'note', 'item': 'printer paper', 'quantity': 1, 'priority': 'low'}

========== FAILURE 2 ==========
INPUT: make a note that I need to water the plants — this is medium priority.
EXPECTED: {'intent': 'note', 'item': 'water the plants', 'quantity': 1, 'priority': 'medium'}
MODEL OUTPUT: {"intent": "note", "item": "I need to water the plants", "quantity": 1, "priority": "medium"}
PARSED OUTPUT: {'intent': 'note', 'item': 'I need to water the plants', 'quantity': 1, 'priority': 'medium'}

========== FAILURE 3 ==========
INPUT: please submit my assignment — this is high priority.
EXPECTED: {'intent': 'task', 'item': 'submit my assignment', 'quantity': 1,

In [20]:
import json
import os
import shutil

# Save metrics
results = {
    "base_model": base_metrics,
    "finetuned_model": ft_metrics,
    "training_examples": len(train),
    "test_examples": len(test),
    "model": MODEL_ID,
    "gpu": torch.cuda.get_device_name(0)
}

with open("task1_results.json", "w") as f:
    json.dump(results, f, indent=2)

# Save predictions
with open("task1_predictions.jsonl", "w") as f:
    for row in base_rows + ft_rows:
        f.write(json.dumps(row) + "\n")

# Save adapter folder as ZIP
shutil.make_archive(
    "qwen_lora_adapter",
    "zip",
    "qwen_lora_adapter"
)

print("All Task 1 files saved!")

All Task 1 files saved!


## Memory estimate for 7B (rough)
Assume 7 billion parameters, decimal GB, FP16 weights, and Adam with FP32 first/second moments.
- Full fine-tuning: weights = 7B × 2 bytes = 14 GB; gradients ≈ 14 GB; Adam moments ≈ 56 GB. Total ≈ 84 GB before activations, temporary buffers and fragmentation (practical peak is higher).
- LoRA: frozen FP16 base weights ≈ 14 GB, plus small adapter weights, adapter gradients/optimizer states, activations and overhead. The base does not need gradients or optimizer states.
- QLoRA: ideal 4-bit base weights = 7B × 0.5 bytes = 3.5 GB, plus quantisation metadata, adapters, activations and runtime overhead. Actual memory exceeds 3.5 GB.

These are estimates, not measured values. Context length, batch size and checkpointing change peak memory. Record three real fine-tuned failures and explain them; if fewer than three occur, report that honestly. Dataset is template-generated, so discuss limited linguistic diversity.